In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.1"

import numpy as np
import matplotlib.pyplot as plt
import particle_cluster as pcl
import jax
import jzfmm
import aegis
import json
from dataclasses import replace

In [ ]:
run = 15
res = np.load(f"logs/particle_sim_{run}.npz")
snap = np.load(
    f"logs/particle_sim_{run}_snapshots.npz"
)["snapshots"]
cfg = pcl.Config.from_json(f"logs/particle_sim_{run}.json")

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

target_initial = pcl.get_particles(res["target_parameters"])
target_final = pcl.simulate(res["target_parameters"], cfg)

axs[0].scatter(
    target_initial.pos[:, 0], target_initial.pos[:, 1],
    s=2, label="target", color="black"
)
axs[1].scatter(
    target_final.pos[:, 0], target_final.pos[:, 1],
    s=2, label="target", color="black"
)

if False:
    initial = pcl.get_particles(res["initial_parameters"])
    initial_final = pcl.simulate(res["initial_parameters"], cfg)

    axs[0].scatter(
        initial.pos[:, 0], initial.pos[:, 1],
        s=1, alpha=0.15, label="step 0",
    )
    axs[1].scatter(
        initial_final.pos[:, 0], initial_final.pos[:, 1],
        s=1, alpha=0.15, label="step 0",
    )

    for snapshot in snap:
        if snapshot["step"] == 0:
            continue

        particles_initial = pcl.get_particles(snapshot["parameters"])
        particles_final = pcl.simulate(snapshot["parameters"], cfg)
        label = f"step {snapshot['step']}"

        axs[0].scatter(
            particles_initial.pos[:, 0],
            particles_initial.pos[:, 1],
            s=1, alpha=0.15, label=label,
        )
        axs[1].scatter(
            particles_final.pos[:, 0],
            particles_final.pos[:, 1],
            s=1, alpha=0.15, label=label,
        )

best_initial = pcl.get_particles(res["best_parameters"])
best_final = pcl.simulate(res["best_parameters"], cfg)

axs[0].scatter(
    best_initial.pos[:, 0], best_initial.pos[:, 1],
    s=2, label="best", alpha=0.5
)
axs[1].scatter(
    best_final.pos[:, 0], best_final.pos[:, 1],
    s=2, label="best", alpha=0.5
)

axs[0].set_title("initial")
axs[1].set_title("final")

for ax in axs:
    ax.set_xlim(-1000, 1000)
    ax.set_ylim(-1000, 1000)
    ax.set_xlabel("x [kpc]")
    ax.set_ylabel("y [kpc]")
    ax.set_aspect("equal")
    ax.legend()

In [ ]:
run = 9
res = np.load(f"logs/particle_ic_{run}.npz")
snap = np.load(f"logs/particle_ic_{run}_snapshots.npz")["snapshots"]

plt.figure(figsize=(6, 6))

target = pcl.get_particles(res["target_parameters"])
plt.scatter(target.pos[:, 0], target.pos[:, 1], s=2, label="target")

initial = pcl.get_particles(res["initial_parameters"])
plt.scatter(
    initial.pos[:, 0], initial.pos[:, 1],
    s=1, alpha=0.15, label="step 0",
)

for snapshot in snap:
    if snapshot["step"] == 0:
        continue
    particles = pcl.get_particles(snapshot["parameters"])
    plt.scatter(
        particles.pos[:, 0],
        particles.pos[:, 1],
        s=1,
        alpha=0.15,
        label=f"step {snapshot['step']}",
    )

best = pcl.get_particles(res["best_parameters"])
plt.scatter(best.pos[:, 0], best.pos[:, 1], s=2, label="best")

plt.xlim(-1000, 1000)
plt.ylim(-1000, 1000)
plt.xlabel("x [kpc]")
plt.ylabel("y [kpc]")
plt.legend()